# В этом ноутбуке обучается финальный ансамбль для предсказания цен автомобилей.

Эксперименты, выбор моделей, веса ансамбля и параметры калибровки были зафиксированы в `02_experiments_model_selection.ipynb`.

Здесь не подбираются гиперпараметры и не создаётся submission. Цель ноутбука — воспроизводимо обучить финальные компоненты на всей обучающей выборке и сохранить артефакты для последующего инференса.

# пути, импорты и папки

In [5]:
# ==============================================================
# 1. PROJECT SETUP
# ==============================================================

from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


# --------------------------------------------------------------
# Ищем корень проекта независимо от текущей рабочей папки.
# --------------------------------------------------------------

current_path = Path.cwd().resolve()

possible_roots = [
    current_path,
    *current_path.parents,
]

PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "data").exists()
        and (path / "models").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта.\n"
        f"Текущая папка: {current_path}\n\n"
        "Ожидалась структура:\n"
        "shift_ml/\n"
        "├── data/\n"
        "├── models/\n"
        "├── notebooks/\n"
        "└── reports/"
    )

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

FINAL_MODEL_DIR = MODELS_DIR / "final_ensemble_12_66"

for directory in [
    PROCESSED_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    FINAL_MODEL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Final model directory:", FINAL_MODEL_DIR)

Project root: C:\temp\shift_ml
Processed data: C:\temp\shift_ml\data\processed
Final model directory: C:\temp\shift_ml\models\final_ensemble_12_66


# Фиксируем финальную конфигурацию

In [6]:
# ==============================================================
# 2. FROZEN FINAL ENSEMBLE CONFIGURATION
# ==============================================================

FINAL_ENSEMBLE_CONFIG = {
    "experiment_name": "final_ensemble_12_66",
    "leaderboard_score": 12.66,
    "submission_name": "submission_v10_conditional_disagree.zip",
    "random_state": RANDOM_STATE,
    "target_column": "Цена",
    "id_column": "car_id",
    "catboost_params": {
        "loss_function": "RMSE",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 8,
        "l2_leaf_reg": 5,
        "random_seed": RANDOM_STATE,
        "verbose": 500,
        "allow_writing_files": False,
    },
    "ensemble_weights": {
        "ridge": 0.196562,
        "v5": 0.019991,
        "v6": 0.0,
        "v8": 0.348682,
        "stats": 0.101857,
        "text": 0.038848,
        "retrieval_k3_pred": 0.132080,
        "v10_stats_v8_pred": 0.161980,
    },
    "disagreement_cut_points": [
        0.04946076,
        0.08297837,
    ],
    "disagreement_multipliers": {
        "0": 0.994,
        "1": 0.980,
        "2": 0.951,
    },
    "retrieval": {
        "k_neighbors": 3,
        "distance_epsilon": 0.15,
        "missing_numeric_penalty": 0.20,
        "missing_categorical_penalty": 0.15,
    },
}

CONFIG_PATH = (
    FINAL_MODEL_DIR
    / "ensemble_config.json"
)

with open(
    CONFIG_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        FINAL_ENSEMBLE_CONFIG,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Saved config:", CONFIG_PATH)

pd.DataFrame(
    {
        "component": list(
            FINAL_ENSEMBLE_CONFIG[
                "ensemble_weights"
            ].keys()
        ),
        "weight": list(
            FINAL_ENSEMBLE_CONFIG[
                "ensemble_weights"
            ].values()
        ),
    }
).sort_values(
    "weight",
    ascending=False,
)

Saved config: C:\temp\shift_ml\models\final_ensemble_12_66\ensemble_config.json


,component,weight
3,v8,0.348682
0,ridge,0.196562
7,v10_stats_v8_pred,0.161980
6,retrieval_k3_pred,0.132080
4,stats,0.101857
5,text,0.038848
1,v5,0.019991
2,v6,0.000000



V8 CatBoost	0.348682	
Основная модель ансамбля. Обучается на технических характеристиках автомобиля и расширенных признаках из названия: нормализованном названии, комплектациях, мощности, приводе, типе двигателя, спортивных и премиальных версиях. Лучше всего улавливает сложные нелинейные зависимости между характеристиками автомобиля и ценой.

Ridge на log1p(Цена) 0.196562
Линейная модель на числовых и категориальных признаках. Она слабее CatBoost отдельно, но делает ошибки другого типа: более стабильно обобщает общие рыночные зависимости и меньше склонна к локальным переобученным решениям. Поэтому её прогноз заметно улучшает ансамбль.

V10: V8 + target statistics 0.161980
CatBoost-модель на признаках V8, дополненных статистиками цены для групп автомобилей: марка + модель, марка + модель + год, нормализованное название и название + год. Статистики строились через cross-fitting, чтобы не допустить утечку таргета. Эта модель лучше использует локальную структуру рынка конкретных моделей и поколений.

Retrieval K=3 0.132080
Прогноз по трём наиболее похожим автомобилям из train. Сходство рассчитывается по марке, модели, году, пробегу, двигателю, коробке, приводу, кузову и комплектации. Retrieval сам по себе слабее CatBoost, но добавляет локальную информацию о реальных аналогах, которой нет в глобальной модели.

CatBoost с target statistics 0.101857
Модель, построенная преимущественно на сглаженных статистиках цены по группам похожих автомобилей. Она дополняет V8 и V10: сильнее опирается на типичную цену конкретной модели, года выпуска или версии.

Text Ridge 0.038848
TF-IDF + Ridge-модель по текстовым полям и названию автомобиля. Она ищет редкие слова, обозначения комплектаций, двигателей и версий, которые могут быть недостаточно хорошо разобраны вручную. Вес небольшой, но компонент улучшает ансамбль за счёт разнообразия ошибок.

V5 CatBoost 0.019991
Более ранняя версия CatBoost с иерархическими признаками из названия. Почти полностью заменена V8, но сохранила небольшой вес, потому что в отдельных случаях даёт полезно отличающийся прогноз.

V6 CatBoost 0.000000
Включалась в подбор весов как проверенный кандидат, но после учёта остальных компонентов не добавила независимого сигнала. Поэтому оптимизация назначила ей нулевой вес, и в финальном прогнозе она фактически не используется.

Ансамбль объединяет:

глобальные закономерности из CatBoost;

более устойчивые линейные зависимости из Ridge;

информацию из текста;

среднюю цену похожих групп автомобилей;

локальные цены ближайших аналогов.

Итоговый прогноз вычисляется как взвешенная сумма прогнозов компонентов:

raw_prediction = (
    0.348682 * v8_pred
    + 0.196562 * ridge_pred
    + 0.161980 * v10_pred
    + 0.132080 * retrieval_pred
    + 0.101857 * stats_pred
    + 0.038848 * text_pred
    + 0.019991 * v5_pred
)

После ансамблирования применяется отдельная calibration по степени расхождения моделей. Если модели сильно расходятся в оценке цены, такой объект считается более сложным, и прогноз дополнительно корректируется вниз. Эта процедура дала устойчивое улучшение на nested OOF-validation и улучшила итоговый leaderboard score до 12.66.

# загрузка финальных данных

Эта ячейка специально проверяет несколько возможных имён файлов, потому что данные по ходу проекта сохранялись в нескольких вариантах.

In [8]:
# ==============================================================
# 3. LOAD PREPARED MODEL DATA
#
# Эти файлы создаются в 01_data_eda_features.ipynb.
# Здесь намеренно не выполняется feature engineering:
# Notebook 3 отвечает только за финальное обучение моделей.
# ==============================================================

TRAIN_MODEL_INPUT_PATH = (
    PROCESSED_DIR
    / "train_model_input_v8.parquet"
)

TEST_MODEL_INPUT_PATH = (
    PROCESSED_DIR
    / "test_model_input_v8.parquet"
)

X_TRAIN_V8_PATH = (
    PROCESSED_DIR
    / "X_train_v8.parquet"
)

X_TEST_V8_PATH = (
    PROCESSED_DIR
    / "X_test_v8.parquet"
)

required_data_paths = {
    "train_model_input_v8.parquet": TRAIN_MODEL_INPUT_PATH,
    "test_model_input_v8.parquet": TEST_MODEL_INPUT_PATH,
    "X_train_v8.parquet": X_TRAIN_V8_PATH,
    "X_test_v8.parquet": X_TEST_V8_PATH,
}

missing_data_files = {
    name: path
    for name, path in required_data_paths.items()
    if not path.exists()
}

if missing_data_files:
    missing_files_text = "\n".join(
        f"— {name}: {path}"
        for name, path in missing_data_files.items()
    )

    raise FileNotFoundError(
        "Не найдены подготовленные данные для финального обучения.\n\n"
        "Сначала нужно полностью выполнить "
        "01_data_eda_features.ipynb, который создаёт "
        "итоговые V8-признаки.\n\n"
        f"Не найдены файлы:\n{missing_files_text}"
    )

train_model_input = pd.read_parquet(
    TRAIN_MODEL_INPUT_PATH
)

test_model_input = pd.read_parquet(
    TEST_MODEL_INPUT_PATH
)

X_train_v8 = pd.read_parquet(
    X_TRAIN_V8_PATH
)

X_test_v8 = pd.read_parquet(
    X_TEST_V8_PATH
)

TARGET_COLUMN = FINAL_ENSEMBLE_CONFIG[
    "target_column"
]

ID_COLUMN = FINAL_ENSEMBLE_CONFIG[
    "id_column"
]

if TARGET_COLUMN not in train_model_input.columns:
    raise KeyError(
        f"В train_model_input нет target: {TARGET_COLUMN}"
    )

if TARGET_COLUMN in test_model_input.columns:
    raise ValueError(
        "В test_model_input не должно быть столбца Цена."
    )

if ID_COLUMN not in train_model_input.columns:
    raise KeyError(
        f"В train_model_input нет идентификатора: {ID_COLUMN}"
    )

if ID_COLUMN not in test_model_input.columns:
    raise KeyError(
        f"В test_model_input нет идентификатора: {ID_COLUMN}"
    )

if list(X_train_v8.columns) != list(X_test_v8.columns):
    raise ValueError(
        "Наборы признаков train и test не совпадают."
    )

if len(train_model_input) != len(X_train_v8):
    raise ValueError(
        "Число строк train_model_input и X_train_v8 не совпадает."
    )

if len(test_model_input) != len(X_test_v8):
    raise ValueError(
        "Число строк test_model_input и X_test_v8 не совпадает."
    )

y_train = train_model_input[
    TARGET_COLUMN
].copy()

train_ids = train_model_input[
    ID_COLUMN
].copy()

test_ids = test_model_input[
    ID_COLUMN
].copy()

print("Train model input:", train_model_input.shape)
print("Test model input:", test_model_input.shape)
print("X_train_v8:", X_train_v8.shape)
print("X_test_v8:", X_test_v8.shape)

print("\nTarget summary:")
display(y_train.describe())

print("\nFirst V8 features:")
display(X_train_v8.head(3))

FileNotFoundError: Не найдены подготовленные данные для финального обучения.

Сначала нужно полностью выполнить 01_data_eda_features.ipynb, который создаёт итоговые V8-признаки.

Не найдены файлы:
— train_model_input_v8.parquet: C:\temp\shift_ml\data\processed\train_model_input_v8.parquet
— test_model_input_v8.parquet: C:\temp\shift_ml\data\processed\test_model_input_v8.parquet
— X_train_v8.parquet: C:\temp\shift_ml\data\processed\X_train_v8.parquet
— X_test_v8.parquet: C:\temp\shift_ml\data\processed\X_test_v8.parquet